# small-llm — train a tiny GPT in Colab

This notebook clones the repo, installs dependencies, downloads a small text corpus, trains a tiny GPT, and samples text from it.

**Tip:** for a GPU, go to *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

## 1. Clone the repo & install

In [ ]:
# If you forked the repo, change the URL below to your fork.
REPO_URL = "https://github.com/lukas04545/llm.git"

import os
if not os.path.isdir("llm"):
    !git clone $REPO_URL llm
%cd llm
!pip install -q torch requests

## 2. Check the device

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Download training data
Downloads Tiny Shakespeare (~1 MB). Falls back to the bundled sample if offline. You can also upload your own `.txt` and point `--data_path` at it.

In [ ]:
!python scripts/prepare_data.py

## 4. Train
On a free T4 GPU you can use a bigger model. On CPU, drop the size and `max_iters` (e.g. defaults).

In [ ]:
import torch
if torch.cuda.is_available():
    !python train.py --device=cuda --n_layer=6 --n_head=6 --n_embd=384 \
        --block_size=256 --batch_size=64 --max_iters=5000
else:
    !python train.py

## 5. Generate text

In [ ]:
!python generate.py --prompt="To be, or not to be" --max_new_tokens=400 --temperature=0.8 --top_k=40

## 6. (Optional) Generate interactively from Python

In [ ]:
import torch
from model import GPT, GPTConfig
from tokenizer import CharTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("out/ckpt.pt", map_location=device)
model = GPT(GPTConfig(**ckpt["model_args"]))
model.load_state_dict(ckpt["model"])
model.to(device).eval()
tok = CharTokenizer.load("out/tokenizer.json")

prompt = "What's in a name?"
idx = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
out = model.generate(idx, max_new_tokens=300, temperature=0.8, top_k=40)
print(tok.decode(out[0].tolist()))